# Water Quality Prediction: Benchmark Notebook 

In this notebook, we demonstrate a basic workflow that serves as a foundation for the challenge. The model has been developed to predict **water quality parameters** using features derived from the **Landsat** and **TerraClimate** datasets. Specifically, four spectral bands — **SWIR22** (Shortwave Infrared 2), **NIR** (Near Infrared), **Green**, and **SWIR16** (Shortwave Infrared 1) — were utilized from Landsat, along with derived spectral indices such as **NDMI** (Normalized Difference Moisture Index) and **MNDWI** (Modified Normalized Difference Water Index). In addition, the **PET** (Potential Evapotranspiration) variable was incorporated from the **TerraClimate** dataset to account for climatic influences on water quality.

The dataset spans a five-year period from **2011 to 2015**. Using **API-based data extraction** methods, both Landsat and TerraClimate features were retrieved directly from the [Microsoft Planetary Computer portal](https://planetarycomputer.microsoft.com).

These combined spectral, index-based, and climatic features were used as predictors in a regression model to estimate three key water quality parameters: **Total Alkalinity (TA)**, **Electrical Conductance (EC)**, and **Dissolved Reactive Phosphorus (DRP)**.

## Environment preparation

In [ ]:
!pip install uv
!uv pip install -r ../requirements.txt 

In [1]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data manipulation and analysis
import numpy as np
import pandas as pd
from IPython.display import display

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

# Geospatial raster data handling with CRS support
import rioxarray as rxr

# Raster operations and spatial windowing
import rasterio
from rasterio.windows import Window

# Feature preprocessing and data splitting
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree

# Machine Learning
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os 

In [ ]:
# Features that don't need to be converted to float, and do not go on model building
not_to_convert_data = [
    'Latitude',
    'Longitude',
    'Sample Date'
]

# Features not included on train
cols_to_drop = [
    'Total Alkalinity',
    'Electrical Conductance',
    'Dissolved Reactive Phosphorus',
    'Sample Date'
]

In [ ]:
# Raw data
water_quality_df = (
    pd.read_csv("../data/raw/water_quality_training_dataset.csv")
)

# Landsat
landsat_train_features = (
    pd.read_csv("../data/processed/landsat_train_features.csv")
)

landsat_val_features = (
    pd.read_csv("../data/processed/landsat_val_features.csv")
)

# Terraclimate
terraclimate_train_features = (
    pd.read_csv("../data/intermediate/terraclimate_features_training.csv")
)

terraclimate_val_features = (
    pd.read_csv("../data/intermediate/terraclimate_features_validation.csv")
)

# Submission
base_submission_df = (
    pd.read_csv("../data/raw/submission_template.csv")
)

In [ ]:
def convert_numeric_features(data):
    """
    Guarantees numeric features in all dfs
    """
    if not isinstance(data, list):
        dfs = [data]
    else:
        dfs = data
        
    for df in dfs:
        for col in df.columns:
            if not col in not_to_convert_data:
                df[col] = df[col].astype(float)

In [ ]:
def create_date_features(df, date_column='Sample Date'):
    
    df = df.copy()
    # Guarantees date
    df[date_column] = pd.to_datetime(df[date_column], dayfirst=True, errors='coerce')
    
    # Extract basic features
    df['month'] = df[date_column].dt.month
    df['year'] = df[date_column].dt.year
    df['quarter'] = df[date_column].dt.quarter
    
    # Ciclic transformation
    # Sen and Cos 'connects' December and January in a 'circle'
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    
    return df

In [5]:
def combine_datasets(main_df, feature_dfs_list):
    """
    Returns a  vertically concatenated dataset.
    You can pass as many datasets you want, inside a list.
    Verifys lenght to avoid errors.
    """
    combined = main_df.copy()
    
    for df_features in feature_dfs_list:
        # Security verification
        if len(combined) != len(df_features):
            print(f"DANGER: different sizes! Main: {len(combined)}, Feature: {len(df_features)}")
            continue
            
        # Removes duplicate columns
        cols_to_use = df_features.columns.difference(combined.columns)

        # Concat
        combined = pd.concat([combined, df_features[cols_to_use]], axis=1)
        
    return combined

## Data transformation

In [6]:
# Conveting numeric features
convert_numeric_features([
    landsat_train_features,
    landsat_val_features,
    landsat_train_features,
    terraclimate_val_features
])

# Combining base data and final data into a single dataset.
# Train
combined_train_df = combine_datasets(
    water_quality_df,
    [landsat_train_features, terraclimate_train_features]
)

# Test
combined_val_df = combine_datasets(
    landsat_val_features,
    [terraclimate_val_features]
)

In [7]:
adjusted_combined_train_df = (
    combined_train_df.fillna(combined_train_df.median(numeric_only=True))
)

adjusted_combined_test_df = (
    combined_val_df.fillna(combined_val_df.median(numeric_only=True))
)

## Model Building

In [9]:
def split_data(x, y, test_size=0.3, random_state=42):
    return train_test_split(x, y, test_size=test_size, random_state=random_state)

def train_model(x_train, y_train):
    """
    XGBRegressor is best than the previous Random Forest because it
    handles NaN by default and dont need scaled data
    """
    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05, # Slow learn
        max_depth=6,
        random_state=42,
        n_jobs=-1 # All power!
    )
    
    model.fit(x_train, y_train)
    return model

def evaluate_model(model, x_test, y_true, dataset_name="Test"):
    """
    model = 'fitted' model with training data
    x_test = missing data
    y_true = real data

    returns the related feature forecasted using the x_test
    and validates it before using on the final database
    """
    # Forecasting
    y_pred = model.predict(x_test)

    # Evaluating model
    r2 = r2_score(y_true, y_pred) # 1.0 is perfect, 0.0 is horrible
    rmse = np.sqrt(mean_squared_error(y_true, y_pred)) # Less is better

    print(f"\n{dataset_name} Evaluation:")
    print(f"R²: {r2:.3f}")
    print(f"RMSE: {rmse:.3f}")

    # Plotting results
    plt.figure(figsize=(8, 4))

    # X axis -> real
    # Y axis -> forecast
    sns.scatterplot(x=y_true, y=y_pred, alpha=0.5)

    # If a dot is above the line -> superestimated
    # If a dot is under the line -> underestimated
    max_val = max(y_true.max(), y_pred.max())
    min_val = min(y_true.min(), y_pred.min())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)

    # Plot
    plt.xlabel("Real")
    plt.ylabel("Forecasted")
    plt.title(f"Real vs Forecasted ({dataset_name}) - R²: {r2:.2f} | RMSE: {rmse:.2f}")
    plt.show()
    
    return y_pred

In [10]:
def run_pipeline(x, y, param_name="Parameter"):
    print(f"\n{'='*60}")
    print(f"Training Model for {param_name}")
    print(f"{'='*60}")
    
    # Split data - needs to be improved (group segregation)
    x_train, x_test, y_train, y_test = split_data(x, y)
    
    # Training model
    model = train_model(x_train, y_train)
    
    # Evaluate (in-sample)
    y_train_pred = evaluate_model(model, x_train, y_train, "Train")
    
    # Evaluate (out-sample)
    y_test_pred = evaluate_model(model, x_test, y_test, "Test")
    
    return model

In [11]:
# In this step, we apply the complete modeling pipeline to each of the three selected water quality parameters:
#     Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus.
    
# The input feature set (`X`) remains the same across all three models, while the target variable (`y`) changes for each parameter. 
# For every parameter, the `run_pipeline()` function is executed, which handles data preprocessing, model training, and both in-sample
# and out-of-sample evaluation. This ensures a consistent workflow and allows for a fair comparison of model performance across different
# water quality indicators.

# Extracts date features
training_df_with_date_features = create_date_features(adjusted_combined_train_df)
testing_df_with_date_features = create_date_features(adjusted_combined_test_df)

# Extracts only desired features
x = training_df_with_date_features.drop(columns=cols_to_drop, errors='ignore')
submission_features = testing_df_with_date_features.drop(columns=cols_to_drop, errors='ignore')

# Targets
y_TA = training_df_with_date_features['Total Alkalinity']
y_EC = training_df_with_date_features['Electrical Conductance']
y_DRP = training_df_with_date_features['Dissolved Reactive Phosphorus']

model_TA = run_pipeline(x, y_TA, "Total Alkalinity")
model_EC = run_pipeline(x, y_EC, "Electrical Conductance")
model_DRP = run_pipeline(x, y_DRP, "Dissolved Reactive Phosphorus")

## Submission

In [20]:
# --- Predicting for Total Alkalinity ---
X_sub_scaled_TA = scaler_TA.transform(submission_val_data)
pred_TA_submission = model_TA.predict(X_sub_scaled_TA)

# --- Predicting for Electrical Conductance ---
X_sub_scaled_EC = scaler_EC.transform(submission_val_data)
pred_EC_submission = model_EC.predict(X_sub_scaled_EC)

# --- Predicting for Dissolved Reactive Phosphorus ---
X_sub_scaled_DRP = scaler_DRP.transform(submission_val_data)
pred_DRP_submission = model_DRP.predict(X_sub_scaled_DRP)

In [21]:
submission_df = pd.DataFrame({
    'Longitude': test_file['Longitude'].values,
    'Latitude': test_file['Latitude'].values,
    'Sample Date': test_file['Sample Date'].values,
    'Total Alkalinity': pred_TA_submission,
    'Electrical Conductance': pred_EC_submission,
    'Dissolved Reactive Phosphorus': pred_DRP_submission
})

In [23]:
#Dumping the predictions into a csv file.
submission_df.to_csv("/tmp/submission_v1.csv", index=False)

session.sql("""
    PUT file:///tmp/submission_v1.csv
    'snow://workspace/USER$.PUBLIC."waterscript-ey-data-science-challenge"/versions/live/submissions/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("Uploaded to notebook environment! Refresh your browser tab to see in sidebar.")